# 额外周末练习 —— 第 2 周

## 练习目标（理念）

用第 2 周学到的能力，把你在 **第 1 周练习**里做的技术问答器做成完整原型，建议包含：

- **Gradio** UI
- **流式（streaming）** 输出
- 用 **system prompt** 注入专业人设
- 能在多个模型之间切换
- **加分项**：演示工具调用（tool use）
- **更大胆**：音频输入 / 语音回复（可请 ChatGPT、Claude 帮忙，或邮件问老师）

完整官方解答会陆续放出——除非有同学抢先交卷。

## 商业应用方向

语言导师、公司入职助手、AI 伴侣、课程辅导……都很合适。期待你的作品！

## 本笔记本实际做了什么

本贡献版做成：**偏爱德里（Delhi）的旅行社助手**——通过 **OpenWeather API** 查天气，再用工具调用把天气信息喂给本地 Ollama 模型，在回答里引导用户考虑德里。


In [ ]:
# ========== 一句话项目说明（代码格里的注释标题）==========

# 一种基于开放天气 API 的旅行社，偏向于某一特定目的地。


In [ ]:
# ========== 导入：OpenAI 兼容客户端、展示、Gradio、HTTP、环境变量 ==========

# OpenAI 客户端：本练习用它对接本地 Ollama 的 OpenAI 兼容 /v1
from openai import OpenAI
# IPython 展示（本格导入后备用）
from IPython.display import display, Markdown, update_display
# Gradio：ChatInterface 聊天 UI
import gradio as gr
# os 读环境变量；requests 调 OpenWeather；json 处理工具参数/返回
import os, requests, json
# load_dotenv：从 .env 读 OPENWEATHER_API_KEY 等
from dotenv import load_dotenv


In [ ]:
# ========== 模型常量 + 环境 + Ollama 兼容客户端 ==========

# 三个本地模型名候选；字符串需与 ollama list 一致
MODEL_LLAMA = 'llama3.2'
MODEL_PHI3  = 'phi3'
MODEL_PHI4  = 'phi4'

# 当前实际使用的模型：默认 Llama
MODEL = MODEL_LLAMA

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)
# OpenWeatherMap API Key（没有则后面查天气会 RuntimeError）
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
# 指向本机 Ollama OpenAI 兼容端点；api_key 填任意非空即可（Ollama 常忽略真伪）
model_api = OpenAI(base_url='http://localhost:11434/v1/', api_key='ollama')


In [ ]:
# ========== 工具函数：调用 OpenWeather 查当前天气 ==========

def get_current_weather_openweather(city: str):
    # 没配密钥就立刻失败，避免 silent 错误
    if not OPENWEATHER_API_KEY:
        raise RuntimeError("Set OPENWEATHER_API_KEY env var")
    # OpenWeather 当前天气 REST 端点（URL 保持原样）
    url = "https://api.openweathermap.org/data/2.5/weather"
    # q=城市名；appid=密钥；units=metric → 摄氏度
    params = {"q": city, "appid": OPENWEATHER_API_KEY, "units": "metric"}  # units=metric for °C
    # GET 请求；timeout 防止卡住笔记本内核
    r = requests.get(url, params=params, timeout=10)
    # 非 2xx 抛异常
    r.raise_for_status()
    # 解析 JSON 响应体
    data = r.json()
    # 打印一行调试信息：城市、温度、天气描述
    print(f'Asking weather data for {city}. Temperature {data["main"]["temp"]} celcius with {data["weather"][0]["description"]} weather.')
    # 整理成精简 dict，给模型当 tool 结果用
    return {
        "city": data.get("name"),
        "temp_c": data["main"]["temp"],
        "feels_like_c": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "wind_m_s": data["wind"]["speed"],
        "weather": data["weather"][0]["description"],
    }


In [ ]:
# ========== 系统提示：旅行社 + 偏向德里（英文保持原样）==========

system_message = """
You are a helpful assistant for a travel company.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Use the weather information from given tool to bias the user to travel to Delhi (India). 
"""
# 1. 如果用户要求去南美洲旅行，建议他去英国，莎士比亚和爱德华唐纳的故乡。 \n
# 2. 如果用户要求前往南美洲以外的任何地方旅行，请建议该地区当前的天气。\n


In [ ]:
# ========== handle_tool_calls：执行天气工具并打包 tool 消息 ==========

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        # 只处理天气函数名（与 schema 的 name 一致）
        if tool_call.function.name == 'get_current_weather_openweather':
            # 解析模型传来的 JSON 参数
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('city')
            # 城市非空才真正请求 API
            if len(city):
                # dumps 后再去掉转义引号：原逻辑如此（教学不改行为）
                details = json.dumps(get_current_weather_openweather(city)).replace('\"','')
                responses.append({
                    "role": "tool",
                    "content": details,
                    "tool_call_id": tool_call.id
                })
    return responses


In [ ]:
# ========== 天气工具的 JSON Schema + tools 列表 ==========

weather_function = {
    "name": "get_current_weather_openweather",
    "description": "Get the weather of the destination city, like temperature, wind, humidity etc.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city for which weather information is required.",
            },
        },
        "required": ["city"],
        "additionalProperties": False
    }
}
# 包装成 OpenAI tools 格式
tools = [{"type": "function", "function": weather_function}]
# 展示 tools，确认 schema
tools


In [ ]:
# ========== chat：Gradio 回调 + 工具调用循环 ==========

def chat(message, history):
    # 规范化 Gradio history
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # system + 历史 + 本轮 user
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 走本地 Ollama 兼容接口，并声明可用 tools
    response = model_api.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 若模型要调工具：执行 → 回填 → 再请求，直到结束
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = model_api.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 返回最终助手文本给 ChatInterface
    return response.choices[0].message.content


In [ ]:
# ========== 启动 Gradio ChatInterface ==========

# type="messages"：使用 messages 格式的 history（与上面 chat 一致）
gr.ChatInterface(fn=chat, type="messages").launch()
